### Installation des dépendances

Configuration adaptée du notebook `LLM setup` fourni dans le modèle de projet pour exécuter [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) et le pipeline RAG sur Google Colab.

- L'environnement Colab est configuré avec un GPU T4 et un runtime Python 3.11.
- La cellule installe les dépendances nécessaires avec `uv`, en fixant les versions de certaines bibliothèques et en ajoutant celles nécessaires au RAG.
- L'installation est ignorée lorsque PandasAI 3.0.0 et les trois bibliothèques RAG sont déjà présents. Après une première installation, un redémarrage de la session Colab est nécessaire.
- Hors Colab, l'installation est ignorée et l'environnement peut être configuré localement avec `uv sync --extra llm --extra rag`.

In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IN_COLAB = "google.colab" in sys.modules


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


# Skipped once the packages are installed, so this cell can be re-run after restarting the session.
if IN_COLAB and installed_version("pandasai") != "3.0.0":
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")

    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])

    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==2.3.3",
        "transformers==5.17.0",
        "accelerate==1.15.0",
        "bitsandbytes==0.50.2",
        "peft==0.21.2",
        "sentence-transformers>=6.1.0",
        "requests",
        "pandasai==3.0.0",
    ]

    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")

### Récupération du dépôt GitHub

Cette cellule clone le dépôt GitHub de notre équipe dans `/content/repo` afin de rendre le package `ift6758` accessible depuis Colab.

- Le clonage est effectué uniquement si le dossier cible n'existe pas déjà.
- Le dossier du dépôt est ajouté à `sys.path` pour permettre l'importation des modules Python du projet.
- Hors Colab, cette étape est ignorée.

In [ ]:
import os

if IN_COLAB:
    from google.colab import userdata

    REPO = "ammou35/LNH-data-science"
    REPO_DIR = "/content/repo"
    BRANCH = "llm-code-generation"

    if not os.path.exists(REPO_DIR):
        subprocess.check_call(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO}", REPO_DIR])

    PROJECT_DIR = os.path.join(REPO_DIR, "data_science_project")
    sys.path.insert(0, PROJECT_DIR)

### Vérification de l'environnement

- Vérification de la version de Python et des principales dépendances.
- Affichage des versions utilisées pour documenter l'environnement et faciliter la reproductibilité des résultats.

In [ ]:
import sys
import torch
import numpy as np
import pandas as pd
from importlib.metadata import version

if sys.version_info[:2] != (3, 11):
    raise RuntimeError("A Python 3.11 runtime is required.")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Transformers:", version("transformers"))
print("Accelerate:", version("accelerate"))
print("PandasAI:", version("pandasai"))

### Vérification du GPU

- Si un GPU est disponible, son nom est affiché.
- Sinon, un message rappelle de sélectionner un runtime GPU T4 dans les paramètres de Google Colab.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).")

### Chargement du modèle Qwen

- Le modèle Qwen3-4B-Instruct-2507 est chargé à l'aide de la fonction `load_model()` du package `ift6758`.
- Lors de la première exécution, environ 8 Go de poids sont téléchargés dans le cache Hugging Face.
- Le périphérique d'exécution et le type de données numériques du modèle sont affichés après le chargement.
- Une première requête sur le hockey permet de vérifier la génération de texte avec le gabarit de conversation de Qwen.

In [ ]:
from ift6758.llm import load_model

model, tokenizer = load_model()
print(f"Loaded on {model.device} ({model.dtype})")

In [ ]:
def ask_qwen(prompt, system="You are a helpful assistant.", max_new_tokens=512):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask_qwen("In one sentence, what is a hat trick in hockey?"))

### Chargement et nettoyage des données de la NHL

Cette cellule récupère les données de la NHL de 2018 à 2020, les regroupe dans un dataframe pandas et calcule la distance des tirs au filet.

In [ ]:
from ift6758.data.nhl import download_seasons
from ift6758.data import season_to_pandas
from ift6758.visualizations.simple_viz import compute_distance

download_seasons(first=2018, last=2020)
shots = pd.concat([season_to_pandas(year) for year in range(2018, 2021)], ignore_index=True)
shots = compute_distance(shots)

display(shots.head())

### Calcul des résultats de référence

Cette cellule calcule les nombres de tirs, de buts et les pourcentages de buts par équipe pour la saison 2018-19. Ces résultats servent à vérifier les trois types de requêtes reprises de la démo.

In [ ]:
expected = shots.loc[shots["season"] == 2018].groupby("team").agg(shots=("event_id", "size"), goals=("is_goal", "sum"))
expected["goal_pct"] = 100 * expected["goals"] / expected["shots"]
display(expected)

### Intégration de Qwen avec PandasAI

Cette cellule relie Qwen à PandasAI pour interroger le dataframe en langage naturel. Le modèle génère le code d'analyse que PandasAI exécute sur les données NHL.

In [ ]:
from pandasai import SmartDataframe
from pandasai.llm.base import LLM

class QwenPandasLLM(LLM):
    calls = 0

    @property
    def type(self):
        return "qwen-local"

    def call(self, instruction, context=None):
        self.calls += 1
        self.last_prompt = instruction.to_string() + (
            "\nUse the existing df = dfs[0]. Complete the calculation in the QUERY. "
            "Replace all TODOs and placeholder text with working Python. "
            "Do not read another file or recreate the sample data."
        )
        self.last_response = ask_qwen(
            self.last_prompt,
            system="You are a data analyst. Return executable pandas code in a fenced python block. "
                   "Follow the requested result dictionary format.",
            max_new_tokens=768,
        )
        return self.last_response

plot_dir = Path("data/plots").resolve()
plot_dir.mkdir(parents=True, exist_ok=True)
pandas_llm = QwenPandasLLM()
sdf = SmartDataframe(shots, config={
    "llm": pandas_llm, "enable_cache": False, "verbose": False,
    "save_logs": False, "max_retries": 1, "use_error_correction_framework": False,
    "save_charts": True, "save_charts_path": str(plot_dir), "open_charts": False,
})
data_context = (
    "This dataframe contains NHL shot-on-goal and goal events from the 2018-19, 2019-20 and 2020-21 seasons. "
    "Each row is one shot event INCLUDING goals. is_goal is Boolean. "
    "The season column contains the starting year (e.g. 2018, 2019 or 2020). "
    "The team column contains NHL team abbreviations. "
    "Missing shot_type values must not exclude rows from overall shot counts. "
    "Use df = dfs[0] directly and filter the requested season. "
    "Do not read another file or invent unavailable information. "
)

### Vérification d'une requête numérique

Cette cellule demande à Qwen le nombre de buts de Montréal en 2018-19, puis compare sa réponse au résultat calculé avec pandas.

In [ ]:
from numbers import Number

count_prompt = data_context + "How many goals did MTL score during the 2018-19 season? Return a number."
count_answer = sdf.chat(count_prompt, output_type="number")
count_code = sdf.last_code_executed
print("Answer:", count_answer)
print("Generated code:\n", count_code)
count_ok = isinstance(count_answer, Number) and count_answer == int(expected.loc["MTL", "goals"])
print("Matches pandas:", count_ok)

### Vérification d'une requête de table

Cette cellule demande à Qwen de regrouper les tirs et les buts par équipe, puis compare le dataframe obtenu aux résultats de référence.

In [ ]:
table_prompt = data_context + (
    "Return a DataFrame with exactly three columns: team, shots (count of rows), "
    "goals (sum of is_goal). Group by team so there is one row per team."
)
table_answer = sdf.chat(table_prompt, output_type="dataframe")
table_code = sdf.last_code_executed
display(table_answer)
print(table_code)
# PandasAI 2 wraps tabular answers in a SmartDataframe.
actual = table_answer.dataframe if isinstance(table_answer, SmartDataframe) else table_answer
try:
    pd.testing.assert_frame_equal(
        actual.set_index("team").sort_index()[["shots", "goals"]], expected[["shots", "goals"]].sort_index(),
        check_dtype=False, check_exact=False, rtol=1e-6, atol=1e-6,
    )
    table_ok = True
except (AssertionError, AttributeError, KeyError, TypeError) as error:
    table_ok = False
    print("Mismatch to investigate:", error)
print("Matches pandas:", table_ok)

### Vérification d'une requête de graphique

Cette cellule demande à Qwen de produire un graphique des buts par équipe. Les valeurs représentées sont examinées à partir des résultats de référence.

In [ ]:
chart_prompt = data_context + (
    "Create a bar chart of total goals for each team: group by team and sum is_goal. "
    "Label the axes Team and Goals. Title: MTL vs CAR — one game. "
    "The output folder already exists; do not create directories."
)
chart_answer = sdf.chat(chart_prompt, output_type="plot")
chart_code = sdf.last_code_executed
if isinstance(chart_answer, str) and chart_answer.endswith(".png") and Path(chart_answer).is_file():
    display(Image(filename=chart_answer))
else:
    print(chart_answer)
print(chart_code)
display(expected[["goals"]])  # Compare these values with the bar heights.

### Bilan des requêtes

Cette cellule regroupe les résultats des trois vérifications et présente le taux de réussite après validation du graphique.

In [ ]:
chart_ok = None  # Replace with True or False after inspecting the chart; an error is False.
evaluation = pd.DataFrame({
    "task": ["MTL goal count", "Team summary", "Goals-by-team chart"],
    "correct": [count_ok, table_ok, chart_ok],
})
display(evaluation)
print("Model calls, including internal repairs:", pandas_llm.calls)
if evaluation["correct"].notna().all():
    print(f"Success rate: {evaluation['correct'].astype(bool).mean():.0%}")
else:
    print("Review the chart before reporting a success rate.")

### 8 Questions d'évaluation

Cette cellule définit huit questions fixes pour évaluer le système, couvrant les regroupements, les pourcentages de buts, les graphiques et une question à laquelle les données ne permettent pas de répondre.

In [ ]:
evaluation_questions = [
    # Strings
    ("Which shot type had the highest goal percentage in the 2018-19 season?", "string"),
    ("Which shot type had the highest goal percentage among shots taken more than 30 feet from the net in the 2019-20 season?", "string"),
    ("Which shot type was the most frequent in the 2019-20 season?", "string"),
    ("How many goals were scored without an assist from another player in the 2020-21 season?", "string"), # Question impossible

    # Tables
    ("How many goals did MTL and TOR each score in the 2020-21 season?", "dataframe"),
    ("Compare the goal percentages on wrist shots for MTL and TOR in the 2018-19 season.", "dataframe"),

    # Graphes
    ("Create a bar chart comparing shot events and goals for each shot type in the 2020-21 season.", "plot"),
    ("Create a graph showing goal percentage by distance from the net in the 2018-19 season, using 5-foot distance bins. Only display percentages for bins containing at least 10 shot events.", "plot"),
]

### Calcul des réponses de référence

Cette cellule calcule les réponses attendues aux 8 questions avec Pandas et les fonctions de l'équipe, afin de les comparer indépendamment aux résultats de Qwen.

In [ ]:
from ift6758.visualizations.simple_viz import get_shots_and_goals, compute_goal_percent

# Q1
shots_2018 = shots.loc[shots["season"] == 2018]
shot_counts, goal_counts = get_shots_and_goals(shots_2018, "shot_type", count=True)
goal_counts = goal_counts.reindex(shot_counts.index, fill_value=0)

q1 = (100 * goal_counts / (shot_counts + goal_counts)).sort_values(ascending=False)

# Q2
shots_2019_far = shots.loc[
    (shots["season"] == 2019)
    & (shots["distance"] > 30)
]

shot_counts, goal_counts = get_shots_and_goals(shots_2019_far, "shot_type", count=True)
goal_counts = goal_counts.reindex(shot_counts.index, fill_value=0)

q2 = (100 * goal_counts / (shot_counts + goal_counts)).sort_values(ascending=False)

# Q3
q3 = shots.loc[shots["season"] == 2019, "shot_type"].value_counts()

# Q4
q4 = "Impossible à déterminer car les passes décisives ne figurent pas dans les données ordonnées."

# Q5
q5 = shots.loc[
    (shots["season"] == 2020)
    & shots["team"].isin(["MTL", "TOR"])
].groupby("team").agg(
    goals=("is_goal", "sum")
)

# Q6
q6 = shots.loc[(shots["season"] == 2018)
    & shots["team"].isin(["MTL", "TOR"])
    & (shots["shot_type"] == "wrist")
].groupby("team").agg(
    shots=("event_id", "size"),
    goals=("is_goal", "sum"),
)

q6["goal_pct"] = 100 * q6["goals"] / q6["shots"]

# Q7
shots_2020 = shots.loc[shots["season"] == 2020]
shot_counts, goal_counts = get_shots_and_goals(shots_2020, "shot_type", count=True)
goal_counts = goal_counts.reindex(shot_counts.index, fill_value=0)

q7 = pd.DataFrame({
    "shots": shot_counts + goal_counts,
    "goals": goal_counts,
})

# Q8
bins = np.arange(0, 200, 5)

q8 = pd.Series(
    compute_goal_percent(
        shots_2018,
        bin_size=5,
        df_columns=bins,
    ),
    index=bins,
    name="goal_pct",
)

references = [q1, q2, q3, q4, q5, q6, q7, q8]

for question, reference in zip(evaluation_questions, references):
    print("\nQuestion:", question[0])
    display(reference)

### Exécution des questions d'évaluation

Cette cellule soumet les huit questions à PandasAI et conserve les réponses ainsi que le code généré par Qwen pour leur vérification.

In [ ]:
answers = []
codes = []

for question, output_type in evaluation_questions:
    try:
        answer = sdf.chat(data_context + question, output_type=output_type)
        code = sdf.last_code_executed
    except Exception as error:
        answer = error
        code = None

    answers.append(answer)
    codes.append(code)

    print("\nQuestion:", question)
    display(answer)
    print("Generated code:\n", code)

### Comparaison des réponses avec les références

Cette cellule présente chaque réponse générée par Qwen avec son résultat de référence, afin de vérifier les valeurs, les regroupements et les graphiques.

In [ ]:
for i, ((question, output_type), answer, reference) in enumerate(
    zip(evaluation_questions, answers, references), start=1
):
    print(f"\nQ{i} — {question}")
    print("\nRéponse de Qwen :")

    if output_type == "plot" and isinstance(answer, str) and Path(answer).is_file():
        display(Image(filename=answer))
    else:
        display(answer)

    print("\nRéférence Pandas :")
    display(reference)

### Table des évaluations

Cette cellule récapitule, pour chaque question, la réussite de la première exécution, l'exactitude de la réponse finale et le nombre de nouvelles tentatives.

In [ ]:
first_attempts = [
    None,   # Q1
    None,   # Q2
    None,   # Q3
    None,   # Q4
    None,   # Q5
    None,   # Q6
    None,   # Q7
    None,   # Q8
]
correct = [
    None,   # Q1
    None,   # Q2
    None,   # Q3
    None,   # Q4
    None,   # Q5
    None,   # Q6
    None,   # Q7
    None,   # Q8
]
retries = [
    None,   # Q1
    None,   # Q2
    None,   # Q3
    None,   # Q4
    None,   # Q5
    None,   # Q6
    None,   # Q7
    None,   # Q8
]

evaluation = pd.DataFrame({
    "task": [question for question, _ in evaluation_questions],
    "first_attempt": first_attempts,
    "correct": correct,
    "retries": retries,
})

display(evaluation)